# Data Validations

## Purpose: 
* Check `Unique Binding Pattern` (`UBP`) table
* Make sure no variance in `expected value` from `SHAP`
* Make sure that `SHAP additivity` principle holds up for randomly sampled rows

In [ ]:
import pandas as pd
pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_columns', 1000)
pd.set_option('display.max_colwidth', 1000)

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import polars as pl 

_=pl.Config.set_tbl_cols(1000)
_=pl.Config.set_tbl_rows(1000)
_=pl.Config.set_tbl_width_chars(10000)
_=pl.Config.set_fmt_str_lengths(10000)

import numpy as np
import sys, yaml, glob

with open("1_metadata_variables.yaml", 'r') as f:
    metadata_variables = yaml.safe_load(f)

## View `Unique Binding Pattern (UBP)` ID Table

In [ ]:
for cell_line in metadata_variables['CELL_LINES']: 

    pl.scan_csv(
        f"../outputs/ubp_ID_tables/{cell_line}_unique_binding_pattern_ID_reference_table.tsv.gz", 
        separator="\t"
    ).head(20).collect()

## No Variance in Expected Value

In [ ]:
for cell_line in metadata_variables["CELL_LINES"]: 
    files = glob.glob(f"../outputs/expected_values/{cell_line}_*.txt")

    f"Number of files for {cell_line}: {len(files)}"

    expected_values = []

    for file in files: 
        with open(file, 'r') as in_file: 
            
            expected_values.append(
                float(in_file.readline().strip())
            )

    pd.Series(expected_values).describe()

## Check Additivity for Randomly Sampled Rows

In [ ]:
import random, gzip, pickle

In [ ]:
seed = random.randint(1, 9000)
seed 
random.seed(seed)

for cell_line in metadata_variables["CELL_LINES"]: 
    files = glob.glob(f"../outputs/flattened_interaction_value_tables/{cell_line}_*")
    files = random.sample(files, 2)

    files

    ubp_id_table_path = f"{metadata_variables['UBP_ID_TABLE_DIR']}/{cell_line}_unique_binding_pattern_ID_reference_table.tsv.gz"
    ubp_id_table = pl.read_csv(
        ubp_id_table_path,
        separator="\t", 
    )
    
    model_hash= metadata_variables["XGBOOST_BEST_MODEL_HASHES"][cell_line]
    model_path = f'{metadata_variables["MODEL_DIR"]}/{model_hash}.pkl.gz'

    # Load the model
    with gzip.open(model_path, 'rb') as f:
        model = pickle.load(f)

    with open(f"../outputs/expected_values/{cell_line}_1-1000_expected_value.txt", 'r') as in_file: 
        expected_value = float(in_file.readline().strip())

    f"Expected value: {expected_value}"
        
    for file_path in files:
        # Read the IPC scan file
        lf = pl.scan_ipc(file_path)
        shap_columns = [col for col in lf.collect_schema().names() if col.endswith('-shap')]
        
        shap_sum = lf.select(shap_columns).with_columns(
            pl.sum_horizontal(shap_columns).alias("shap_sum")
        ).select("shap_sum").collect()["shap_sum"].to_numpy()
        assert shap_sum.ndim == 1, "shap_sum is not a 1D array"

        splitter = file_path.split("/")[-1].split("-")
        start_id = int(splitter[0].split("_")[-1])
        end_id = int(splitter[1].split("_")[0])

        start_id
        end_id
        
        X = ubp_id_table.filter(
            (pl.col("Unique Binding Pattern ID #") >= start_id) & 
            (pl.col("Unique Binding Pattern ID #") <= end_id)      
        ).select(
            [col for col in ubp_id_table.columns if col.endswith("_binding")]
        ).to_pandas()
            
        # Predict with output_margin=True
        preds = model.predict(X, output_margin=True)
        
        # Calculate distribution: horizontal sum of SHAP columns + expected_value - prediction
        distribution = shap_sum + expected_value - preds
        distribution = pd.Series(distribution,)

        distribution.describe()
        distribution.head()        